# Notebook 08 Part 1: Spark SQL, Temporary Views and SQL Queries

## 1. What Is Spark SQL?

Spark SQL allows us to process PySpark DataFrames using SQL queries.

```python
spark.sql("""
    SELECT customer_name, amount
    FROM orders
    WHERE amount > 5000
""")
```

Simple meaning:

```text
Spark SQL lets us run SQL on distributed Spark data.
```

## 2. DataFrame API vs Spark SQL

The same task can be written in two styles.

| DataFrame API | Spark SQL |
|---|---|
| Uses Python methods | Uses SQL syntax |
| Good for programmatic pipelines | Good for SQL users |
| Easy to combine with Python logic | Easy for analysts |
| Optimized by Spark | Also optimized by Spark |

### DataFrame API

```python
df.filter(df.amount > 5000).select("customer_name", "amount")
```

### Spark SQL

```sql
SELECT customer_name, amount
FROM orders
WHERE amount > 5000
```

## 3. Windows Local Setup

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

## 4. Create SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark_SQL_Temporary_Views") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

## 5. Create Sample DataFrames

In [3]:
orders_data = [
    (101, 1, "Laptop", 50000.0, "Delhi"),
    (102, 2, "Mobile", 30000.0, "Mumbai"),
    (103, 1, "Tablet", 25000.0, "Delhi"),
    (104, 3, "Laptop", 55000.0, "Pune"),
    (105, 4, "Monitor", 18000.0, "Mumbai"),
    (106, 5, "Mobile", None, "Delhi")
]

orders_columns = ["order_id", "customer_id", "product", "amount", "city"]
orders_df = spark.createDataFrame(orders_data, orders_columns)
orders_df.show()

+--------+-----------+-------+-------+------+
|order_id|customer_id|product| amount|  city|
+--------+-----------+-------+-------+------+
|     101|          1| Laptop|50000.0| Delhi|
|     102|          2| Mobile|30000.0|Mumbai|
|     103|          1| Tablet|25000.0| Delhi|
|     104|          3| Laptop|55000.0|  Pune|
|     105|          4|Monitor|18000.0|Mumbai|
|     106|          5| Mobile|   NULL| Delhi|
+--------+-----------+-------+-------+------+



In [4]:
customers_data = [
    (1, "Amit", "Premium"),
    (2, "Priya", "Standard"),
    (3, "Rahul", "Premium"),
    (4, "Sneha", "Standard"),
    (6, "Karan", "Premium")
]

customers_columns = ["customer_id", "customer_name", "customer_type"]
customers_df = spark.createDataFrame(customers_data, customers_columns)
customers_df.show()

+-----------+-------------+-------------+
|customer_id|customer_name|customer_type|
+-----------+-------------+-------------+
|          1|         Amit|      Premium|
|          2|        Priya|     Standard|
|          3|        Rahul|      Premium|
|          4|        Sneha|     Standard|
|          6|        Karan|      Premium|
+-----------+-------------+-------------+



## 6. Create Temporary Views

A DataFrame must be registered as a view before it can be queried through SQL.

```python
df.createOrReplaceTempView("view_name")
```

In [5]:
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")

A temporary view exists only during the current SparkSession.

## 7. Run the First SQL Query

In [7]:
result_df = spark.sql("""
    SELECT *
    FROM orders
    where amount > 50000
""")

result_df.show()

+--------+-----------+-------+-------+----+
|order_id|customer_id|product| amount|city|
+--------+-----------+-------+-------+----+
|     104|          3| Laptop|55000.0|Pune|
+--------+-----------+-------+-------+----+



`spark.sql()` returns another DataFrame.

In [9]:
spark.sql("""
    SELECT
        COUNT(*) AS total_orders,
        SUM(amount) AS total_sales,
        AVG(amount) AS average_sales,
        MIN(amount) AS minimum_sales,
        MAX(amount) AS maximum_sales
    FROM orders
""").show()

+------------+-----------+-------------+-------------+-------------+
|total_orders|total_sales|average_sales|minimum_sales|maximum_sales|
+------------+-----------+-------------+-------------+-------------+
|           6|   178000.0|      35600.0|      18000.0|      55000.0|
+------------+-----------+-------------+-------------+-------------+



## 23. Create or Replace a View

`createOrReplaceTempView()` creates a new view or replaces an existing view with the same name.

In [ ]:
high_value_orders_df = orders_df.filter(orders_df.amount >= 30000)
high_value_orders_df.createOrReplaceTempView("high_value_orders")

spark.sql("""
    SELECT *
    FROM high_value_orders
""").show()

## 24. List and Remove Temporary Views

In [ ]:
spark.catalog.listTables()

In [ ]:
spark.catalog.dropTempView("high_value_orders")

#### Inspect the SQL Execution Plan

In [ ]:
spark.sql("""
    SELECT city, SUM(amount) AS total_sales
    FROM orders
    GROUP BY city
""").explain()

You may see operations such as:

```text
Scan
Project
HashAggregate
Exchange
```

`Exchange` usually indicates shuffle caused by `GROUP BY`.

## 28. Common Errors

| Error | Correction |
|---|---|
| View not registered | Run `createOrReplaceTempView()` first |
| Wrong column name | Check `df.columns` or `printSchema()` |
| String without quotes | Use `WHERE city = 'Delhi'` |
| Aggregate condition in `WHERE` | Use `HAVING` |

In [ ]:
spark.stop()